# JSON

Nous allons créer un serveur un peu special pour le notebook. Une boucle `while True:` ne rendant jamais la main, notre serveur prendrait toutes les ressources du notebook, sans permettre à d'autres programmes de s'exécuter. Ici on utilise le module `asyncio` qui evite ces blocages. Ce rend le code un peu plus compliqué, mais on ne va pas le décrire ici, nous en reparlerons dans la suite du MOOC.

Lancez l'execution de la cellule, le serveur va afficher que le serveur est démarré.

In [ ]:
import asyncio

# 1. Définition du protocole Serveur
class EchoServerProtocol(asyncio.DatagramProtocol):
    def connection_made(self, transport):
        self.transport = transport
        print("🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)")

    def datagram_received(self, data, addr):
        message = data.decode()
        print(f"📥 Serveur a reçu : '{message}' de {addr}")

# 2. Fonction pour lancer le serveur en arrière-plan
async def start_background_server():
    loop = asyncio.get_running_loop()
    
    # create_datagram_endpoint crée le socket et l'attache à la boucle
    transport, protocol = await loop.create_datagram_endpoint(
        lambda: EchoServerProtocol(),
        local_addr=('127.0.0.1', 33033)
    )
    
    # On retourne le transport pour pouvoir le fermer plus tard si besoin
    return transport

# 3. Lancement de la tâche serveur
# On stocke 'server_transport' dans une variable globale pour pouvoir l'arrêter
server_transport = await start_background_server()

🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)


📥 Serveur a reçu : 'Hello' de ('127.0.0.1', 61896)
📥 Serveur a reçu : '19.902652127443503' de ('127.0.0.1', 61896)
📥 Serveur a reçu : ' 19.964916818789018, 1000.1126197826824, 29.128148831424948 ' de ('127.0.0.1', 61896)
📥 Serveur a reçu : '[19.964916818789018, 29.128148831424948, 1000.1126197826824]' de ('127.0.0.1', 61896)


Maintenant un premier programme qui ne marche pas, car on n'a pas mis en place la sérialisation et que l'on souhaite envoyer une variable en mémoire au travers du réseau.

In [5]:
from virtual_sensor import virtual_sensor 
import time
import socket

temperature = virtual_sensor(start=20, variation = 0.1)
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

t = temperature.read_value()
print (f"🌡️ Température lue : {t} °C")
s.sendto (t, ("127.0.0.1", 33033))

🌡️ Température lue : 19.91866462579085 °C


TypeError: a bytes-like object is required, not 'float'

Aie, ca fait mal, on peut pas envoyer ce type d'objet via le réseau, par contre une chaime de caractère ca passe. 

In [4]:
s.sendto ("Hello".encode(), ("127.0.0.1", 33033))

5

remarquez l'appel à `encode()` pour convertir la chaîne en bytes. `sendto`est têtu, il ne veut que des octets, pas des caractères qui peuvent avoir des codages bizarres quand on sort des caractères ASCII.

In [5]:
t = temperature.read_value()
s.sendto (str(t).encode(), ("127.0.0.1", 33033))


18

On peut aussi envoyer plusieurs valeurs dans un même message, il suffit de transformer la chaîne de caractères que l'on envoie.

In [6]:
pressure    = virtual_sensor(start=1000, variation = 1) 
humidity    = virtual_sensor(start=30, variation = 3, min=20, max=80) 

t = temperature.read_value ()
p = pressure.read_value ()
h = humidity.read_value ()
msg = f" {t}, {p}, {h} "
s.sendto ( msg.encode () , ( "127.0.0.1" , 33033))

59

Fastoche pour émettre, par contre côté récepteur c'est la galère, il faut analyser la séquence reçue, par exemple: 

```
' 19.929924227176862, 999.5896343856912, 30.6004757327658 ' 
```

Vous auriez envie de faire ça?

Faut mieux demander à un module python, comme JSON de sérailiser ou de désérialiser les données. 

In [7]:
import json

j = json.dumps([t, h, p])

s.sendto ( j.encode() , ( "127.0.0.1", 33033))

60

Le serveur reçoit quelque chose du genre:

```
[19.929924227176862, 30.6004757327658, 999.5896343856912]
```

On va regarder comment améliorer notre serveur pour comprendre cette chaîne de caractères. Dans un premier temps, il faut tuer la coroutine que l'on a créée dans la première cellule de ce notebook.

In [ ]:
server_transport.close()

Et le relancer avec un analyse des données reçues en JSON. Revenez deux cellules en arrière pour émettre des données. 

In [9]:
import json

server_transport.close() # pour etre sûr de fermer avant de relancer

# 1. Définition du protocole Serveur
class EchoServerProtocol(asyncio.DatagramProtocol):
    def connection_made(self, transport):
        self.transport = transport
        print("🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)")

    def datagram_received(self, data, addr):
        message = data.decode()
        data = json.loads(message)
        print(f"données: {data[0]} °C, {data[1]} %HR, {data[2]} hPa de {addr}")

# 2. Fonction pour lancer le serveur en arrière-plan
async def start_background_server():
    loop = asyncio.get_running_loop()
    
    # create_datagram_endpoint crée le socket et l'attache à la boucle
    transport, protocol = await loop.create_datagram_endpoint(
        lambda: EchoServerProtocol(),
        local_addr=('127.0.0.1', 33033)
    )
    
    # On retourne le transport pour pouvoir le fermer plus tard si besoin
    return transport

# 3. Lancement de la tâche serveur
# On stocke 'server_transport' dans une variable globale pour pouvoir l'arrêter
server_transport = await start_background_server()

🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)


données: 20.063040540500047 °C, 27.50317868904771 %HR, 999.5288900287984 hPa de ('127.0.0.1', 61896)


Ok, vous voyez que la partie difficile de comprendre la chaîne de caractères a été pris en charge par le module JSON, après le `loads` les données sont en mémoire et peuvent être manipulées facilement. Vous pourriez modifier le code du serveur pour qu'il affiche la moyenne. 

Pour vous aider, le programma dans la cellule suivante, envoie des données périodiquement.

In [10]:
async def client_minimal():
    
    while True:
        t = temperature.read_value()
        h = humidity.read_value()
        p = pressure.read_value()   
        
        donnees = [t, h, p ]
        
        j = json.dumps(donnees)
        s.sendto(j.encode(), ("127.0.0.1", 33033))
        
        print(f"Envoyé : {j}")
        
        await asyncio.sleep(10)

await client_minimal()

Envoyé : [20.063040540500047, 27.50317868904771, 999.5288900287984]


CancelledError: 

Maitnenant utilisez vos talents de programmeur pour modifier le programme serveur pour qu'il affiche la moyenne. 

# CBOR



Bon, on tue tous nos process pour avoir quelques chose de propre. Le client vous devez l'achever avec vos mains en cliquant sur le petit carré à gauche qui dénote une exécution. Le serveur peut être tué avec la formule magique utulisant `close()`.

A la place on va utiliser cbor.

Lancer les deux cellules ci-dessous.

In [1]:
#code server
import cbor2 as cbor
import asyncio 

#server_transport.close() # pour etre sûr de fermer avant de relancer

# 1. Définition du protocole Serveur
class EchoServerProtocol(asyncio.DatagramProtocol):
    def connection_made(self, transport):
        self.transport = transport
        print("🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)")

    def datagram_received(self, data, addr):
        data = cbor.loads(data)
        print(f"données: {data[0]} °C, {data[1]} %HR, {data[2]} hPa de {addr}")

# 2. Fonction pour lancer le serveur en arrière-plan
async def start_background_server():
    loop = asyncio.get_running_loop()
    
    # create_datagram_endpoint crée le socket et l'attache à la boucle
    transport, protocol = await loop.create_datagram_endpoint(
        lambda: EchoServerProtocol(),
        local_addr=('127.0.0.1', 33033)
    )
    
    # On retourne le transport pour pouvoir le fermer plus tard si besoin
    return transport

# 3. Lancement de la tâche serveur
# On stocke 'server_transport' dans une variable globale pour pouvoir l'arrêter
server_transport = await start_background_server()

🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)


données: 20.05189079938232 °C, 31.05696128477384 %HR, 999.3161578971125 hPa de ('127.0.0.1', 52024)
données: 19.979733573853927 °C, 30.11616801718644 %HR, 999.5130837171658 hPa de ('127.0.0.1', 52024)
données: 19.918321736035555 °C, 27.511915271662506 %HR, 998.9842540144135 hPa de ('127.0.0.1', 52024)
données: 19.832135277624324 °C, 29.492602994120865 %HR, 999.5496408769085 hPa de ('127.0.0.1', 52024)
données: 19.852695780806865 °C, 28.791671709908574 %HR, 999.1226322036886 hPa de ('127.0.0.1', 52024)
données: 19.8386943281102 °C, 28.72682097688152 %HR, 998.5503578043356 hPa de ('127.0.0.1', 52024)
données: 19.827534841104036 °C, 27.683191931884362 %HR, 999.4960644466556 hPa de ('127.0.0.1', 52024)
données: 19.795699914300453 °C, 25.270051641413897 %HR, 999.5634262867765 hPa de ('127.0.0.1', 52024)
données: 19.764106288233943 °C, 23.312667811811433 %HR, 999.6986438021557 hPa de ('127.0.0.1', 52024)
données: 19.841620714503232 °C, 23.725175627498324 %HR, 999.1839217204126 hPa de ('127.0

In [ ]:
#code client 
import asyncio
import socket
import cbor2 as cbor
from virtual_sensor import virtual_sensor

temperature = virtual_sensor(start=20, variation = 0.1)
pressure    = virtual_sensor(start=1000, variation = 1) 
humidity    = virtual_sensor(start=30, variation = 3, min=20, max=80) 

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

async def client_minimal():
    
    while True:
        t = temperature.read_value()
        h = humidity.read_value()
        p = pressure.read_value()   
        
        donnees = [t, h, p ]
        
        j = cbor.dumps(donnees)
        s.sendto(j, ("127.0.0.1", 33033))
        
        print(f"Envoyé : {j.hex()}")
        
        await asyncio.sleep(10)

await client_minimal()

Envoyé : 83fb403405aef77da3dffb403d03de80d89157fb408f3f78f43e6b22
Envoyé : 83fb403411b9c0054e7ffb403d99adf073e534fb408f38b8c7932939
Envoyé : 83fb403415c41b31c98dfb403cc5de42d8c5aefb408f367c2da0a94a
Envoyé : 83fb403407bf9aba24defb403e8ce6b86d5f08fb408f361e622c03a9
Envoyé : 83fb4033f6fca563f9d2fb403dd5dd02e888f9fb408f320b71b28906
Envoyé : 83fb403401ff022ba120fb4040195bee7b883bfb408f3098fe1edd01
Envoyé : 83fb403419d3d047291afb403f63ec359a3418fb408f352334e53ec4
Envoyé : 83fb4034185208d9ff6dfb4040f787fac48922fb408f3bed8419ff63


Si on compare la version utilisant JSON et celle utilisant CBOR, on constate que :
* le code est sensiblement le même, on fait appel aux méthodes loads et dumps (bien que ce qui est produit n'est plus une chaîne de caractères, mais un bytes, les operations de conversion avec `encode` ne sont plus nécessaire.
* la taille du message est moindre avec CBOR (28 octets contre 56 caractères avec JSON). Par contrre si on regarde la structure du CBOR (en tulisant le site https://cbor.me par exemple) on constate que le format que la représentation des nombres est assez grande, car les valeurs sont codées en nombre flottant:
``` 
83                     # array(3)
   FB 4033D7883576E9CF # primitive(4626278222252141007)
   FB 403A259DEF46773B # primitive(4628052927331071803)
   FB 408F4D419D7F797D # primitive(4652021884307601789)
```


In [ ]:
#code client 
import asyncio
import socket
import cbor2 as cbor
from virtual_sensor import virtual_sensor

temperature = virtual_sensor(start=20, variation = 0.1)
pressure    = virtual_sensor(start=1000, variation = 1) 
humidity    = virtual_sensor(start=30, variation = 3, min=20, max=80) 

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

async def client_minimal():
    
    while True:
        t = temperature.read_value()
        h = humidity.read_value()
        p = pressure.read_value()   
        
        donnees = [int(t*100), int(h*100), int(p*100)]
        
        j = cbor.dumps(donnees)
        s.sendto(j, ("127.0.0.1", 33033))
        
        print(f"Envoyé : {j.hex()}")
        
        await asyncio.sleep(10)

await client_minimal()